In [8]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\keert\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\keert\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\keert\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [9]:
df = pd.read_csv("IMDB Dataset.csv")

In [10]:
print("Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())
print("\nSample Review:\n", df['review'][0])

Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Review:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death sta

In [11]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)
df['clean_text'] = df['review'].apply(preprocess_text)
df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch 1 oz episod youll hoo...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [12]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
X = df['clean_text']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [14]:
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [15]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    }

In [18]:
results_bow = {}
LogisticRegression(max_iter=1000)
results_bow['Logistic Regression'] = train_and_evaluate(LogisticRegression(), X_train_bow, X_test_bow, y_train, y_test)
results_bow['Naive Bayes'] = train_and_evaluate(MultinomialNB(), X_train_bow, X_test_bow, y_train, y_test)
results_bow['Decision Tree'] = train_and_evaluate(DecisionTreeClassifier(), X_train_bow, X_test_bow, y_train, y_test)
pd.DataFrame(results_bow).T

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.8728,0.867226,0.882715,0.874902
Naive Bayes,0.8463,0.853167,0.839452,0.846254
Decision Tree,0.7172,0.724103,0.708871,0.716406


In [19]:
results_tfidf = {}
results_tfidf['Logistic Regression'] = train_and_evaluate(LogisticRegression(max_iter=1000), X_train_tfidf, X_test_tfidf, y_train, y_test)
results_tfidf['Naive Bayes'] = train_and_evaluate(MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test)
results_tfidf['Decision Tree'] = train_and_evaluate(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf, y_train, y_test)
pd.DataFrame(results_tfidf).T

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.8856,0.874735,0.902163,0.888238
Naive Bayes,0.8502,0.846681,0.858107,0.852356
Decision Tree,0.7156,0.724759,0.702322,0.713364


Observations:

1. TF-IDF generally performs better than Bag of Words because it gives importance to rare but meaningful words.
2. Logistic Regression achieved the highest accuracy compared to Naive Bayes and Decision Tree.
3. Naive Bayes performs well for text data but may slightly underperform compared to Logistic Regression.
4. Decision Tree tends to overfit and gives lower performance.

Best Model:
Logistic Regression with TF-IDF features.

Trade-offs:
- TF-IDF: Better accuracy but slightly more computation.
- BoW: Faster but less informative.
- Logistic Regression: Balanced and reliable.
- Decision Tree: Easy to interpret but less accurate.